In [ ]:

# 2️⃣ Load dữ liệu Silver
silver_path = "dev.silver_stream.bronze_streaming_events"
df_silver = spark.read.format("delta").load(silver_path)
display(df_silver.limit(5))

In [ ]:
def get_spark():
    """Khởi tạo Spark session (dùng trên Databricks)."""
    return SparkSession.builder.appName("EventsGold").getOrCreate()

# Khởi tạo Spark
spark = get_spark()


In [ ]:
# --------------------------------------------
# 3️⃣ Gold Layer 1: Visitor Profile Table
# --------------------------------------------
visitor_profile_df = df_silver.groupBy("visitorid").agg(
    count(when(col("event") == "view", True)).alias("total_viewcount"),
    countDistinct(when(col("event") == "view", col("itemid"))).alias("unique_items_viewed"),
    count(when(col("event") == "addtocart", True)).alias("total_addtocart"),
    (count(when(col("event") == "transaction", True)) > 0).cast("int").alias("bought_something")
)
display(visitor_profile_df.orderBy(col("total_viewcount").desc()).limit(20))

In [ ]:
# --------------------------------------------
# 4️⃣ Gold Layer 2: Customer Journey Table
# --------------------------------------------
window_spec = Window.partitionBy("visitorid", "itemid").orderBy("event_time")
customer_journey_df = df_silver.withColumn("event_order", row_number().over(window_spec))
display(customer_journey_df.filter(col("visitorid")=="102019").orderBy("event_time"))

In [ ]:
# --------------------------------------------
# 5️⃣ Gold Layer 3: Item Co-occurrence / Recommendation
# --------------------------------------------
transaction_df = df_silver.filter(col("event")=="transaction")
item_cooccurrence_df = transaction_df.groupBy("transactionid").agg(
    collect_list("itemid").alias("items_bought_together")
)
display(item_cooccurrence_df.limit(20))

In [ ]:

# 6️⃣ Phân tích nhanh gợi ý sản phẩm
# Ví dụ: xem item 302422 thì khách trước mua kèm item nào
item_example = "302422"
co_items = item_cooccurrence_df.filter(col("items_bought_together").contains(item_example))
display(co_items)